# Exploração raw — tancagem

Notebook unificado: **perfil exploratório**, **qualidade/chave lógica** e **inventário temporal** dos CSV brutos.

| Seção | Objetivo |
|-------|----------|
| 1. Perfil | Schema, tipos, nulos e domínios categóricos (snapshot recente) |
| 2. Qualidade | Duplicatas, chave primária candidata, CNPJ e agregação por instalação |
| 3. Piloto temporal | Inventário de todos os arquivos baixados (linhas, m³) |

**Pré-requisito:** `py estudos/tancagem-abastecimento/pipelines/download_raw.py`

In [2]:
import re
import sys
from pathlib import Path

import pandas as pd

NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from _paths import RAW_DIR, REPO_ROOT

SAMPLE = RAW_DIR / "2026" / "janeiro.csv"
KEY = ["Data", "CodInstalacao", "Tag", "GrupoDeProdutos"]
CAT_COLS = ["Segmento", "DetalheInstalacao", "GrupoDeProdutos", "TipoDaUnidade", "Uf"]

if not SAMPLE.exists():
    raise FileNotFoundError(f"Execute download_raw.py. Ausente: {SAMPLE}")

def as_id_str(series: pd.Series) -> pd.Series:
    """Identificadores (CNPJ etc.) — texto, sem notação científica."""
    out = series.astype("string")
    return out.str.replace(r"\.0$", "", regex=True)


df = pd.read_csv(SAMPLE, encoding="utf-8", dtype={"Cnpj": "string"})
df["Cnpj"] = as_id_str(df["Cnpj"])

print(f"Snapshot: {SAMPLE.relative_to(REPO_ROOT)}")
print(f"Linhas: {len(df):,} | Colunas: {list(df.columns)}")
print(f"Cnpj dtype: {df['Cnpj'].dtype}")

Snapshot: data\raw\tancagem-abastecimento\2026\janeiro.csv
Linhas: 14,517 | Colunas: ['Data', 'NomeEmpresarial', 'Uf', 'Municipio', 'Cnpj', 'CodInstalacao', 'Segmento', 'DetalheInstalacao', 'Tag', 'TipoDaUnidade', 'GrupoDeProdutos', 'TancagemM3']
Cnpj dtype: string


## 1. Perfil exploratório

In [3]:
df.head(3)

,Data,NomeEmpresarial,Uf,Municipio,Cnpj,CodInstalacao,Segmento,DetalheInstalacao,Tag,TipoDaUnidade,GrupoDeProdutos,TancagemM3
0,2026-01-31,2 IRMAOS PRODUTOS DE PETROLEO LTDA,SP,CAMPINAS,43544287000123,1066748.0,BASES DO RAMO DE TRR,EXCLUSIVA,TQ 01,TANQUE,DERIVADOS E BIOCOMBUSTÍVEIS,20.0
1,2026-01-31,2 IRMAOS PRODUTOS DE PETROLEO LTDA,SP,CAMPINAS,43544287000123,1066748.0,BASES DO RAMO DE TRR,EXCLUSIVA,TQ 02,TANQUE,DERIVADOS E BIOCOMBUSTÍVEIS,20.0
2,2026-01-31,2 IRMAOS PRODUTOS DE PETROLEO LTDA,SP,CAMPINAS,43544287000123,1066748.0,BASES DO RAMO DE TRR,EXCLUSIVA,TQ 05,TANQUE,DERIVADOS E BIOCOMBUSTÍVEIS,20.0


In [4]:
df.info()

print("=== Numérico (medidas) ===")
display(df[["TancagemM3"]].describe().T)

print("=== Categórico / identificador (sem estatísticas numéricas para CNPJ) ===")
id_and_cat = [c for c in df.columns if c != "TancagemM3"]
display(df[id_and_cat].describe(include=["object", "string"]).T)

<class 'pandas.DataFrame'>
RangeIndex: 14517 entries, 0 to 14516
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Data               14517 non-null  str    
 1   NomeEmpresarial    14465 non-null  str    
 2   Uf                 14465 non-null  str    
 3   Municipio          14465 non-null  str    
 4   Cnpj               14465 non-null  string 
 5   CodInstalacao      14465 non-null  float64
 6   Segmento           14465 non-null  str    
 7   DetalheInstalacao  14465 non-null  str    
 8   Tag                14465 non-null  str    
 9   TipoDaUnidade      14465 non-null  str    
 10  GrupoDeProdutos    14465 non-null  str    
 11  TancagemM3         14465 non-null  float64
dtypes: float64(2), str(9), string(1)
memory usage: 3.3 MB
=== Numérico (medidas) ===


,count,mean,std,min,25%,50%,75%,max
TancagemM3,14465.0,4137.948151,9361.175164,0.0,99.0,825.0,4176.0,94412.0


=== Categórico / identificador (sem estatísticas numéricas para CNPJ) ===


,count,unique,top,freq
Data,14517,53,2026-01-31,14465
NomeEmpresarial,14465,1041,PETROLEO BRASILEIRO S/A,1143
Uf,14465,27,SP,3921
Municipio,14465,784,SANTOS,815
Cnpj,14465,1724,33000167008862,261
Segmento,14465,11,BASES DO RAMO DE COMBUSTÍVEIS,2629
DetalheInstalacao,14465,12,EXCLUSIVA,5725
Tag,14465,5947,01,379
TipoDaUnidade,14465,3,TANQUE,13073
GrupoDeProdutos,14465,4,DERIVADOS E BIOCOMBUSTÍVEIS,11093


In [5]:
for col in CAT_COLS:
    print(f"\n=== {col} ({df[col].nunique()} únicos) ===")
    print(df[col].value_counts(dropna=False).head(15).to_string())


=== Segmento (11 únicos) ===
Segmento
BASES DO RAMO DE COMBUSTÍVEIS           2629
TERMINAL                                2626
BASES DO RAMO DE TRR                    2401
INSTALAÇÃO PRODUTORA DE ETANOL          2201
INSTALAÇÃO PRODUTORA DE BIODIESEL       1638
REFINARIA                               1510
BASES DO RAMO DE LIQUEFEITOS            1287
PÓLO DE PROCESSAMENTO DE GÁS NATURAL      75
BASES DO RAMO DE AVIAÇÃO                  66
NaN                                       52
PLANTA DE CPQ                             21
PLANTA DE FORMULADOR                      11

=== DetalheInstalacao (12 únicos) ===
DetalheInstalacao
EXCLUSIVA                               5725
PLANTA DE PRODUCAO DE ETANOL            2201
MARÍTIMO                                1819
PLANTA DE PRODUÇÃO DE BIODIESEL         1638
REFINARIA                               1510
TERRESTRE                                687
COMPARTILHADA                            658
FLUVIAL                                   88
PÓLO

In [6]:
print("Data — distintos:", df["Data"].nunique())
print(df["Data"].value_counts().head())
print("\nTancagemM3 — min/max:", df["TancagemM3"].min(), df["TancagemM3"].max())
print(f"Soma nacional (snapshot): {df['TancagemM3'].sum():,.0f} m³")

Data — distintos: 53
Data
2026-01-31                                                                                                                                                                                                                     14465
2026-01-31,"D'PADUA - DESTILACAO, PRODUCAO, AGROINDUSTRIA E COMERCIO S/A",PB,RIO TINTO,06312488000179,1175560,INSTALAÇÃO PRODUTORA DE ETANOL,PLANTA DE PRODUCAO DE ETANOL,TANQUE 02,TANQUE,DERIVADOS E BIOCOMBUSTÍVEIS,3000        1
2026-01-31,"D'PADUA - DESTILACAO, PRODUCAO, AGROINDUSTRIA E COMERCIO S/A",PB,RIO TINTO,06312488000179,1175560,INSTALAÇÃO PRODUTORA DE ETANOL,PLANTA DE PRODUCAO DE ETANOL,TANQUE 04,TANQUE,DERIVADOS E BIOCOMBUSTÍVEIS,3000        1
2026-01-31,"D'PADUA - DESTILACAO, PRODUCAO, AGROINDUSTRIA E COMERCIO S/A",PB,RIO TINTO,06312488000179,1175560,INSTALAÇÃO PRODUTORA DE ETANOL,PLANTA DE PRODUCAO DE ETANOL,TANQUE 05,TANQUE,DERIVADOS E BIOCOMBUSTÍVEIS,3000        1
2026-01-31,"D'PADUA - DESTILACAO, PRODUCAO, AGROINDUSTRIA 

## 2. Qualidade e chave lógica

Hipótese de chave: `Data` + `CodInstalacao` + `Tag` + `GrupoDeProdutos`

In [7]:
n = len(df)
n_unique = df[KEY].drop_duplicates().shape[0]
dup = df[df.duplicated(KEY, keep=False)].sort_values(KEY)

print(f"Linhas: {n:,}")
print(f"Chaves únicas ({' + '.join(KEY)}): {n_unique:,}")
print(f"Duplicatas na chave: {n - n_unique:,}")
dup.head(10)

Linhas: 14,517
Chaves únicas (Data + CodInstalacao + Tag + GrupoDeProdutos): 14,517
Duplicatas na chave: 0


,Data,NomeEmpresarial,Uf,Municipio,Cnpj,CodInstalacao,Segmento,DetalheInstalacao,Tag,TipoDaUnidade,GrupoDeProdutos,TancagemM3


In [8]:
print("Nulos por coluna:")
print(df.isna().sum())

cnpj = df["Cnpj"].dropna()
cnpj_len = cnpj.str.len()
print("\nCNPJ (texto) — comprimento (moda):", cnpj_len.mode().iloc[0])
print(cnpj_len.value_counts().head())
print("\nCNPJ — amostra de valores:")
print(cnpj.head(5).to_list())

print("\nTancagemM3 <= 0:", (df["TancagemM3"] <= 0).sum())

Nulos por coluna:
Data                  0
NomeEmpresarial      52
Uf                   52
Municipio            52
Cnpj                 52
CodInstalacao        52
Segmento             52
DetalheInstalacao    52
Tag                  52
TipoDaUnidade        52
GrupoDeProdutos      52
TancagemM3           52
dtype: int64

CNPJ (texto) — comprimento (moda): 14
Cnpj
14    14465
Name: count, dtype: Int64

CNPJ — amostra de valores:
['43544287000123', '43544287000123', '43544287000123', '43544287000123', '43544287000123']

TancagemM3 <= 0: 28


In [9]:
by_inst = df.groupby(["Data", "CodInstalacao"], as_index=False)["TancagemM3"].sum()
print("Instalações no snapshot:", len(by_inst))
print("Top 5 instalações por m³:")
by_inst.nlargest(5, "TancagemM3")

Instalações no snapshot: 1727
Top 5 instalações por m³:


,Data,CodInstalacao,TancagemM3
310,2026-01-31,1032153.0,2848849.0
307,2026-01-31,1032149.0,2821332.0
327,2026-01-31,1032197.0,2057932.0
311,2026-01-31,1032154.0,1792111.0
754,2026-01-31,1045563.0,1601658.0


## 3. Piloto temporal (inventário de arquivos)

Diagnóstico por arquivo — orienta integração histórica. Não consolida a série.

In [10]:
files = sorted(RAW_DIR.rglob("*.csv"))
print(f"{len(files)} arquivos CSV em {RAW_DIR.relative_to(REPO_ROOT)}")

36 arquivos CSV em data\raw\tancagem-abastecimento


In [11]:
rows = []
for path in files:
    try:
        part = pd.read_csv(path, encoding="utf-8")
        rows.append(
            {
                "arquivo": str(path.relative_to(RAW_DIR)),
                "linhas": len(part),
                "colunas": len(part.columns),
                "data_distintas": part["Data"].nunique() if "Data" in part.columns else None,
                "soma_m3": part["TancagemM3"].sum() if "TancagemM3" in part.columns else None,
            }
        )
    except Exception as e:
        rows.append({"arquivo": str(path.relative_to(RAW_DIR)), "erro": str(e)})

inv = pd.DataFrame(rows)
inv

# Evolução de linhas ao tempo (por arquivo -> mês)
MONTHS_PT = {
    "janeiro": 1,
    "fevereiro": 2,
    "marco": 3,
    "março": 3,
    "abril": 4,
    "maio": 5,
    "junho": 6,
    "julho": 7,
    "agosto": 8,
    "setembro": 9,
    "outubro": 10,
    "novembro": 11,
    "dezembro": 12,
}


def infer_ym(arquivo: str) -> tuple[int | None, int | None]:
    s = arquivo.lower().replace("\\", "/")

    # Prefer pasta yyyy/mes.csv
    m = re.match(r"^(\d{4})/([^/]+)\.csv$", s)
    if m:
        year = int(m.group(1))
        month_name = m.group(2)
        month = MONTHS_PT.get(month_name)
        return year, month

    # Fallback: ano no caminho e mês no nome do arquivo
    year = None
    m_year = re.search(r"(19\d{2}|20\d{2})", s)
    if m_year:
        year = int(m_year.group(1))

    month = None
    for name, num in MONTHS_PT.items():
        if name in s:
            month = num
            break

    return year, month


inv_plot = inv.dropna(subset=["linhas"]).copy()
inv_plot[["ano", "mes"]] = inv_plot["arquivo"].apply(lambda a: pd.Series(infer_ym(str(a))))
inv_plot = inv_plot.dropna(subset=["ano", "mes"]).copy()
inv_plot["periodo"] = pd.to_datetime(
    inv_plot["ano"].astype(int).astype(str)
    + "-"
    + inv_plot["mes"].astype(int).astype(str).str.zfill(2)
    + "-01"
)

by_period = inv_plot.groupby("periodo", as_index=False)["linhas"].sum().sort_values("periodo")

ax = by_period.plot(x="periodo", y="linhas", kind="line", figsize=(10, 4), legend=False)
ax.set_title("Tancagem — evolução de linhas (inventário por arquivo)")
ax.set_xlabel("Período")
ax.set_ylabel("Linhas")
ax.grid(True, alpha=0.3)

NameError: name 're' is not defined

In [ ]:
other = [p for p in RAW_DIR.rglob("*") if p.is_file() and p.suffix.lower() not in {".csv"}]
for p in other:
    print(p.relative_to(RAW_DIR), f"({p.stat().st_size / 1024:.1f} KB)")